<a href="https://colab.research.google.com/github/Squirrelcoding/green-space-vs-heat/blob/main/Green_Space_vs_Heat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
%pip install gdal

In [3]:
!pip3 install OSMPythonTools
!pip3 install gdal

In [4]:
import ee

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='poopnet-4fb22')

In [5]:
# TODO: Write some code that gets all of the boundaries of a parking lot given its ID or something
from OSMPythonTools.api import Api
api = Api()


In [6]:
import geemap

In [7]:
dataset = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate(
    '2025-05-01', '2025-06-01'
)


# Applies scaling factors.
def apply_scale_factors(image):
  optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
  thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
  return image.addBands(optical_bands, None, True).addBands(
      thermal_bands, None, True
  )


dataset = dataset.map(apply_scale_factors)

visualization = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0.0,
    'max': 0.3,
}

m = geemap.Map()
m.set_center(-114.2579, 38.9275, 8)
m.add_layer(dataset, visualization, 'True Color (432)')
m

Map(center=[38.9275, -114.2579], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

In [8]:
naip_dataset = ee.ImageCollection('USDA/NAIP/DOQQ').filter(ee.Filter.date('2023-01-01', '2024-01-01'))

true_color = naip_dataset.select(['R', 'G', 'B'])

true_color_vis = {
  min: 0,
  max: 255,
};

m = geemap.Map()

m.setCenter(-73.9958, 40.7278, 15)
m.addLayer(true_color, true_color_vis, 'True Color')

m

Map(center=[40.7278, -73.9958], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

Generating URL ...
Please wait ...
Data downloaded to /content/ndvi_colored.tif
ndvi_colored


In [69]:
import os
from osgeo import gdal
from PIL import Image
import numpy as np


# south, west = 44.969979, -93.285288
# north, east = 44.971068, -93.282541
south, west = 44.853759, -93.246084
north, east = 44.855447, -93.244995

roi = ee.Geometry.Rectangle([west, south, east, north])

# Mosaic and calculate NDVI
mosaic = naip_dataset.mosaic()
ndvi = mosaic.normalizedDifference(['N', 'R']).rename('NDVI')

ndvi = ndvi.updateMask(ndvi.gt(0))

vis_params = {
    'min': -1,
    'max': 1,
    'palette': ['black', 'gray', 'white', 'lightgreen', 'green', 'darkgreen']
}

ndvi_vis = ndvi.updateMask(ndvi.gt(0))


# Visualize NDVI to an RGB image
ndvi_rgb = ndvi.visualize(**vis_params).clip(roi)

geemap.ee_export_image(
    ndvi_rgb,
    filename='ndvi_colored.tif',
    scale=1,
    region=roi
)

def tif_to_png(filename: str | os.PathLike):
  pre = filename.split(".")[0]
  print(pre)
  tif = gdal.Open(filename)
  arr = tif.ReadAsArray().transpose(1, 2, 0) # Convert CHW to HWC
  img = Image.fromarray(arr.astype(np.uint8))
  os.remove(filename)
  img.save(pre + '.png')

tif_to_png("ndvi_colored.tif")


img = Image.open("ndvi_colored.png").convert("RGB")
img_np = np.array(img)

r, g, b = img_np[..., 0], img_np[..., 1], img_np[..., 2]

# green should dominate red and blue
green_mask = (g > r) & (g > b)

greenness = (g - np.maximum(r, b)).astype(np.float32)
greenness[~green_mask] = 0

total_greenness = g.sum()

height, width = img_np.shape[:2]
total_pixels = height * width

average_greenness = g.sum() / total_pixels

print("Total greenness:", total_greenness)
print("Average greenness per green pixel:", average_greenness)


Generating URL ...
Please wait ...
Data downloaded to /content/ndvi_colored.tif
ndvi_colored
Total greenness: 4118
Average greenness per green pixel: 0.17859311301934253
